In [17]:
import pandas as pd
import numpy as np
import re
import joblib
import os


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from IPython.display import display
import ipywidgets as widgets

In [18]:
from google.colab import drive
import pandas as pd

# Mount drive
drive.mount('/content/drive')

# File path
file_path = '/content/drive/MyDrive/Scam Dataset/phishing.csv'

# Load dataset
df = pd.read_csv(file_path)

# Count number of records in each class
print(df['label'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
label
1    55914
0    39996
Name: count, dtype: int64


In [19]:
df = pd.read_csv('/content/drive/MyDrive/Scam Dataset/phishing.csv')
print(df.head())

                                              domain   ranking  isIp  valid  \
0                               www.voting-yahoo.com  10000000     0      0   
1         www.zvon.org/xxl/WSDL1.1/Output/index.html    194914     0      1   
2  tecportais.com/file-security-update-infonfmati...  10000000     0      0   
3                bima.astro.umd.edu/nemo/linuxastro/      7001     0      0   
4  huarui-tec.com/js/?us.battle.net/login/en/?ref...  10000000     0      1   

   activeDuration  urlLen  is@  isredirect  haveDash  domainLen  \
0               0      20    0           0         1         20   
1            7305      42    0           0         0         12   
2               0     155    0           0         0         14   
3               0      35    0           0         0         18   
4             730      79    0           0         1         14   

   nosOfSubdomain  label  
0               2      1  
1               2      0  
2               1      1  
3             

In [20]:
# File path
file_path = '/content/drive/MyDrive/Scam Dataset/phishing.csv'

# Load dataset
df = pd.read_csv(file_path)

# Show first rows
print(df.head())
print(df.shape)

                                              domain   ranking  isIp  valid  \
0                               www.voting-yahoo.com  10000000     0      0   
1         www.zvon.org/xxl/WSDL1.1/Output/index.html    194914     0      1   
2  tecportais.com/file-security-update-infonfmati...  10000000     0      0   
3                bima.astro.umd.edu/nemo/linuxastro/      7001     0      0   
4  huarui-tec.com/js/?us.battle.net/login/en/?ref...  10000000     0      1   

   activeDuration  urlLen  is@  isredirect  haveDash  domainLen  \
0               0      20    0           0         1         20   
1            7305      42    0           0         0         12   
2               0     155    0           0         0         14   
3               0      35    0           0         0         18   
4             730      79    0           0         1         14   

   nosOfSubdomain  label  
0               2      1  
1               2      0  
2               1      1  
3             

In [21]:
# Preprocessing function for URLs
def preprocess_url(url):
    url = str(url).lower()
    url = re.sub(r'https?://', '', url)
    url = re.sub(r'www\.', '', url)
    url = re.sub(r'[^a-z0-9]', ' ', url)
    return url

# Clean URL column
df['URL_clean'] = df['domain'].apply(preprocess_url)

# -------- TEXT FEATURE --------
url_text = df['URL_clean']

# -------- NUMERICAL FEATURES --------
numerical_features = df[
    [
        'ranking',
        'isIp',
        'valid',
        'activeDuration',
        'urlLen',
        'is@',
        'isredirect',
        'haveDash',
        'domainLen',
        'nosOfSubdomain'
    ]
]

# Labels
y = df['label']

# Convert URL text into TF-IDF vectors
vectorizer = TfidfVectorizer()
url_text_vec = vectorizer.fit_transform(url_text)

# Combine TF-IDF text features with numerical features
from scipy.sparse import hstack
X = hstack([url_text_vec, numerical_features])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Save model and vectorizer
joblib.dump(model, "phishing_model_hybrid.pkl")
joblib.dump(vectorizer, "vectorizer_hybrid.pkl")

Accuracy: 0.8396934626212074

Classification Report:

              precision    recall  f1-score   support

           0       0.81      0.79      0.80      7810
           1       0.86      0.87      0.87     11372

    accuracy                           0.84     19182
   macro avg       0.83      0.83      0.83     19182
weighted avg       0.84      0.84      0.84     19182



['vectorizer_hybrid.pkl']

In [22]:
# Preprocessing function for URLs
def preprocess_url(url):
    url = str(url).lower()
    url = re.sub(r'https?://', '', url)
    url = re.sub(r'www\.', '', url)
    url = re.sub(r'[^a-z0-9]', ' ', url)
    return url

# Clean URL column
df['URL_clean'] = df['domain'].apply(preprocess_url)

# Features and labels
X = df['URL_clean']
y = df['label']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert URLs into numerical vectors
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Predictions
y_pred = model.predict(X_test_vec)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Save model
joblib.dump(model, "phishing_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

Accuracy: 0.9635595871129183

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.97      0.96      7810
           1       0.98      0.96      0.97     11372

    accuracy                           0.96     19182
   macro avg       0.96      0.96      0.96     19182
weighted avg       0.96      0.96      0.96     19182



['vectorizer.pkl']

In [23]:
# Load saved model and vectorizer
model = joblib.load("phishing_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

In [24]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, model.predict(vectorizer.transform(X_test)))

In [25]:
import ipywidgets as widgets
from IPython.display import display, HTML
import re

# ==========================================
# TITLE SECTION
# ==========================================
title = widgets.HTML(
    value="""
    <h2 style='text-align:center; color:darkblue;'>
        Scam / Genuine URL Detection System
    </h2>
    <p style='text-align:center;'>
        Paste any URL below and click <b>Check URL</b>
    </p>
    """
)

# ==========================================
# INPUT BOX
# ==========================================
url_input = widgets.Text(
    placeholder='Paste URL here...',
    description='URL:',
    layout=widgets.Layout(width='80%')
)

# ==========================================
# BUTTONS
# ==========================================
check_button = widgets.Button(
    description='Check URL',
    button_style='info',
    layout=widgets.Layout(width='150px', height='40px')
)

clear_button = widgets.Button(
    description='Clear',
    button_style='warning',
    layout=widgets.Layout(width='100px', height='40px')
)

# ==========================================
# OUTPUT AREA
# ==========================================
output = widgets.Output()

# ==========================================
# URL VALIDATION FUNCTION
# ==========================================
def is_valid_url(url):
    pattern = re.compile(
        r'^(https?:\/\/)?'          # optional http:// or https://
        r'([a-zA-Z0-9-]+\.)+'       # domain
        r'[a-zA-Z]{2,}'             # extension
        r'(\/.*)?$'                 # optional path
    )
    return re.match(pattern, url) is not None

# ==========================================
# PREDICTION FUNCTION
# ==========================================
def predict_url(btn):
    with output:
        output.clear_output()

        url = url_input.value.strip()

        # Empty input
        if url == "":
            display(HTML("""
            <div style='padding:15px; border-radius:10px; background:#fff3cd;'>
                <h3 style='color:red;'>⚠ Please enter a URL</h3>
            </div>
            """))
            return

        # Invalid URL input
        if not is_valid_url(url):
            display(HTML("""
            <div style='padding:15px; border-radius:10px; background:#fff3cd;'>
                <h3 style='color:orange;'>⚠ Invalid Input</h3>
                <p>Please enter a valid URL.</p>
                <p>Example: <b>https://google.com</b></p>
            </div>
            """))
            return

        # Preprocess and vectorize
        cleaned_url = preprocess_url(url)
        vectorized_url = vectorizer.transform([cleaned_url])

        # Predict
        prediction = model.predict(vectorized_url)[0]

        # Confidence score
        confidence = model.predict_proba(vectorized_url)[0]
        score = max(confidence) * 100

        # Output result
        if prediction == 1:
            display(HTML(f"""
            <div style='padding:15px; border-radius:10px; background:#ffdddd;'>
                <h3 style='color:red;'>⚠ Scam / Phishing URL Detected</h3>
                <p><b>URL:</b> {url}</p>
                <p><b>Confidence:</b> {score:.2f}%</p>
            </div>
            """))
        else:
            display(HTML(f"""
            <div style='padding:15px; border-radius:10px; background:#ddffdd;'>
                <h3 style='color:green;'>✓ Genuine / Safe URL</h3>
                <p><b>URL:</b> {url}</p>
                <p><b>Confidence:</b> {score:.2f}%</p>
            </div>
            """))

# ==========================================
# CLEAR FUNCTION
# ==========================================
def clear_interface(btn):
    url_input.value = ""
    with output:
        output.clear_output()

# ==========================================
# BUTTON ACTIONS
# ==========================================
check_button.on_click(predict_url)
clear_button.on_click(clear_interface)

# ==========================================
# DISPLAY INTERFACE
# ==========================================
button_box = widgets.HBox([check_button, clear_button])

display(title, url_input, button_box, output)

HTML(value="\n    <h2 style='text-align:center; color:darkblue;'>\n        Scam / Genuine URL Detection System…

Text(value='', description='URL:', layout=Layout(width='80%'), placeholder='Paste URL here...')

Output()